In [15]:
import cv2
import numpy as np

def heat_to_green_yellow_red(gray: np.ndarray) -> np.ndarray:
    """
    Convert grayscale heat values to:
    low    -> green
    medium -> yellow
    high   -> red

    Input:  grayscale uint8 image [0..255]
    Output: BGR uint8 image for OpenCV
    """
    x = gray.astype(np.float32) / 255.0

    b = np.zeros_like(x)
    g = np.zeros_like(x)
    r = np.zeros_like(x)

    # 0.0 -> 0.5 : green to yellow
    mask1 = x <= 0.5
    t1 = x[mask1] / 0.5
    r[mask1] = 255.0 * t1
    g[mask1] = 255.0

    # 0.5 -> 1.0 : yellow to red
    mask2 = x > 0.5
    t2 = (x[mask2] - 0.5) / 0.5
    r[mask2] = 255.0
    g[mask2] = 255.0 * (1.0 - t2)

    color = np.stack([b, g, r], axis=-1)  # BGR for OpenCV
    return np.clip(color, 0, 255).astype(np.uint8)


def overlay_heatmap(
    base_path: str,
    heatmap_path: str,
    output_path: str = "overlay_fixed.png",
    max_opacity: float = 0.95,
    threshold: int = 20,
    gamma: float = 1.6
):
    """
    base_path: screenshot / background image
    heatmap_path: black-white heatmap
    output_path: result file
    max_opacity: opacity of brightest heat areas
    threshold: values <= threshold become fully transparent
    gamma: >1.0 makes weak values more transparent
    """

    base = cv2.imread(base_path, cv2.IMREAD_COLOR)
    heat = cv2.imread(heatmap_path, cv2.IMREAD_GRAYSCALE)

    if base is None:
        raise FileNotFoundError(f"Could not load base image: {base_path}")
    if heat is None:
        raise FileNotFoundError(f"Could not load heatmap: {heatmap_path}")

    if heat.shape[:2] != base.shape[:2]:
        heat = cv2.resize(
            heat,
            (base.shape[1], base.shape[0]),
            interpolation=cv2.INTER_LINEAR
        )

    # keep original values, do NOT normalize
    heat_f = heat.astype(np.float32) / 255.0

    # black / near-black becomes transparent
    heat_f[heat <= threshold] = 0.0

    # reduce weak glow
    if gamma != 1.0:
        heat_f = np.power(heat_f, gamma)

    # colorize only visible intensity values
    heat_vis = (heat_f * 255).astype(np.uint8)
    heat_color = heat_to_green_yellow_red(heat_vis).astype(np.float32)

    # opacity from intensity
    alpha = (heat_f * max_opacity)[..., None]

    result = (
        base.astype(np.float32) * (1.0 - alpha) +
        heat_color * alpha
    )

    result = np.clip(result, 0, 255).astype(np.uint8)
    cv2.imwrite(output_path, result)
    print(f"Saved result to: {output_path}")



if __name__ == "__main__":
    image = "0a0c95.jpg"
    overlay_heatmap(
        base_path=f"./dataset/images/{image}",
        heatmap_path=f"./dataset/heatmaps_1s/{image}",   # replace with your actual heatmap file
        output_path="overlay_fixed.png",
        max_opacity=0.8,
        threshold=0,
        gamma=0.5
    )

Saved result to: overlay_fixed.png
